# NexusTrade — 3-Month Comparison Analyzer
## Historical Pattern Analysis for Entry Timing

Vergelijk prijsdata, insider events en rallies over 3 maanden om entry timing te optimaliseren.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from typing import List, Dict, Tuple, Optional

class ThreeMonthAnalyzer:
    """
    Analyseert 3 maanden prijsdata met insider events en news catalysts
    om rally patterns en entry timing te beoordelen.
    """
    
    def __init__(self):
        self.months_data = {}  # {month_name: {price_data, insider_events, news}}
        self.current_price = None
        
    def add_price_data(self, month: str, 
                       high: float, low: float, close: float,
                       perf_pct: float, avg_volume: int,
                       notable_events: Optional[str] = None):
        """
        Voeg maandelijkse prijsdata toe
        
        Args:
            month: Maandnaam (bijv. "January 2026")
            high: Hoogste prijs van de maand
            low: Laagste prijs van de maand
            close: Slotkoers einde maand
            perf_pct: Performance percentage
            avg_volume: Gemiddeld volume
            notable_events: Belangrijke gebeurtenissen (optioneel)
        """
        self.months_data[month] = {
            'high': high,
            'low': low,
            'close': close,
            'perf_pct': perf_pct,
            'avg_volume': avg_volume,
            'range_pct': ((high - low) / low) * 100,
            'notable_events': notable_events,
            'insider_events': [],
            'news_events': []
        }
        
    def add_insider_event(self, month: str, date: str, 
                          shares: int, price: float, 
                          entity: str, action: str = "SELL"):
        """
        Voeg insider trading event toe
        
        Args:
            month: Maandnaam waarin event plaatsvond
            date: Datum van event
            shares: Aantal aandelen
            price: Prijs per aandeel
            entity: Insider naam/entiteit
            action: BUY of SELL (default SELL)
        """
        if month in self.months_data:
            value = shares * price
            self.months_data[month]['insider_events'].append({
                'date': date,
                'action': action,
                'shares': shares,
                'price': price,
                'value': value,
                'entity': entity
            })
    
    def add_news_event(self, month: str, date: str, 
                       headline: str, price_before: float, 
                       price_after: float):
        """
        Voeg news catalyst toe
        
        Args:
            month: Maandnaam
            date: Datum van news
            headline: Nieuwskop
            price_before: Prijs voor news
            price_after: Prijs na news (einde dag of hoogste punt)
        """
        if month in self.months_data:
            impact_pct = ((price_after - price_before) / price_before) * 100
            self.months_data[month]['news_events'].append({
                'date': date,
                'headline': headline,
                'price_before': price_before,
                'price_after': price_after,
                'impact_pct': impact_pct
            })
    
    def get_monthly_summary(self) -> pd.DataFrame:
        """
        Genereer overzicht van alle maanden
        
        Returns:
            DataFrame met maandelijkse statistieken
        """
        summary = []
        for month, data in self.months_data.items():
            insider_count = len(data['insider_events'])
            insider_volume = sum(e['shares'] for e in data['insider_events'])
            news_count = len(data['news_events'])
            
            summary.append({
                'Month': month,
                'High': data['high'],
                'Low': data['low'],
                'Close': data['close'],
                'Perf %': data['perf_pct'],
                'Range %': round(data['range_pct'], 2),
                'Avg Volume': data['avg_volume'],
                'Insider Sells': insider_count,
                'Shares Sold': insider_volume,
                'News Events': news_count
            })
        
        return pd.DataFrame(summary)
    
    def identify_key_levels(self) -> Dict[str, List[float]]:
        """
        Identificeer belangrijke S/R levels uit 3-maanden data
        
        Returns:
            Dictionary met support en resistance levels
        """
        all_highs = [data['high'] for data in self.months_data.values()]
        all_lows = [data['low'] for data in self.months_data.values()]
        
        # Unieke highs en lows als resistance/support
        resistance_levels = sorted(set(all_highs), reverse=True)
        support_levels = sorted(set(all_lows))
        
        # Bereken 3-maanden gemiddeldes
        avg_high = np.mean(all_highs)
        avg_low = np.mean(all_lows)
        
        return {
            'resistance': resistance_levels,
            'support': support_levels,
            'avg_high': round(avg_high, 4),
            'avg_low': round(avg_low, 4),
            'range_midpoint': round((max(all_highs) + min(all_lows)) / 2, 4)
        }
    
    def analyze_insider_correlation(self) -> Dict[str, any]:
        """
        Analyseer correlatie tussen insider selling en prijsactie
        
        Returns:
            Dictionary met insider selling analysis
        """
        insider_analysis = []
        
        for month, data in self.months_data.items():
            for event in data['insider_events']:
                # Check prijs beweging na insider sell
                insider_analysis.append({
                    'month': month,
                    'date': event['date'],
                    'entity': event['entity'],
                    'sell_price': event['price'],
                    'month_close': data['close'],
                    'month_low': data['low'],
                    'price_dropped_to_low': event['price'] > data['low']
                })
        
        if not insider_analysis:
            return {'status': 'No insider events found'}
        
        df = pd.DataFrame(insider_analysis)
        drops_after_sell = df['price_dropped_to_low'].sum()
        
        return {
            'total_insider_sells': len(df),
            'price_dropped_after': drops_after_sell,
            'correlation_strength': f"{(drops_after_sell/len(df)*100):.1f}% of sells preceded drops",
            'avg_sell_price': round(df['sell_price'].mean(), 4),
            'details': df
        }
    
    def generate_rally_pattern_analysis(self) -> Dict[str, any]:
        """
        Analyseer rally patronen over 3 maanden
        
        Returns:
            Rally pattern statistieken
        """
        rallies = []
        
        for month, data in self.months_data.items():
            if data['perf_pct'] > 0:
                # Bereken rally magnitude
                rally_size = data['perf_pct']
                subsequent_crash = False
                
                # Check of er een crash volgde
                months_list = list(self.months_data.keys())
                current_idx = months_list.index(month)
                if current_idx < len(months_list) - 1:
                    next_month = months_list[current_idx + 1]
                    if self.months_data[next_month]['perf_pct'] < -20:
                        subsequent_crash = True
                
                rallies.append({
                    'month': month,
                    'rally_pct': rally_size,
                    'high_reached': data['high'],
                    'ended_at': data['close'],
                    'gave_back_pct': ((data['high'] - data['close']) / data['high']) * 100,
                    'crash_followed': subsequent_crash
                })
        
        if not rallies:
            return {'status': 'No rallies found in 3-month period'}
        
        df = pd.DataFrame(rallies)
        
        return {
            'total_rallies': len(df),
            'avg_rally_size': round(df['rally_pct'].mean(), 2),
            'avg_giveback': round(df['gave_back_pct'].mean(), 2),
            'crash_follow_rate': f"{(df['crash_followed'].sum()/len(df)*100):.1f}%",
            'details': df
        }
    
    def assess_entry_timing(self, current_price: float) -> Dict[str, any]:
        """
        Beoordeel of current price een goed entry punt is binnen 3M context
        
        Args:
            current_price: Huidige marktprijs
            
        Returns:
            Entry timing assessment
        """
        self.current_price = current_price
        
        # Verzamel alle prijzen uit 3 maanden
        all_prices = []
        for data in self.months_data.values():
            all_prices.extend([data['high'], data['low'], data['close']])
        
        # Bereken percentiel positie
        percentile = (sum(1 for p in all_prices if p <= current_price) / len(all_prices)) * 100
        
        # Bereken afstand tot key levels
        levels = self.identify_key_levels()
        nearest_support = max([s for s in levels['support'] if s < current_price], default=min(levels['support']))
        nearest_resistance = min([r for r in levels['resistance'] if r > current_price], default=max(levels['resistance']))
        
        support_distance = ((current_price - nearest_support) / current_price) * 100
        resistance_distance = ((nearest_resistance - current_price) / current_price) * 100
        
        # Bepaal timing quality
        if percentile <= 25:
            timing = "EXCELLENT (bottom 25% of 3M range)"
        elif percentile <= 50:
            timing = "GOOD (below 3M midpoint)"
        elif percentile <= 75:
            timing = "FAIR (above midpoint)"
        else:
            timing = "POOR (top 25% of range)"
        
        return {
            'current_price': current_price,
            'percentile_position': round(percentile, 1),
            'timing_quality': timing,
            'nearest_support': nearest_support,
            'support_distance_pct': round(support_distance, 2),
            'nearest_resistance': nearest_resistance,
            'resistance_distance_pct': round(resistance_distance, 2),
            'risk_reward_ratio': round(resistance_distance / support_distance, 2) if support_distance > 0 else None,
            '3m_range': {
                'low': min(levels['support']),
                'high': max(levels['resistance']),
                'midpoint': levels['range_midpoint']
            }
        }

def calculate_overshoot_probability(target: float, resistance_level: float, 
                                     historical_rallies: List[Dict]) -> Dict[str, any]:
    """
    Bereken waarschijnlijkheid dat prijs door resistance schiet bij rally
    
    Args:
        target: Doel prijs (bijv. breakeven)
        resistance_level: Belangrijkste resistance
        historical_rallies: Lijst van eerdere rally data
        
    Returns:
        Overshoot analysis
    """
    if not historical_rallies:
        return {'error': 'No historical rally data provided'}
    
    # Analyseer eerdere rally behavior
    overshoots = 0
    total_rallies = len(historical_rallies)
    
    for rally in historical_rallies:
        if 'high_reached' in rally and 'ended_at' in rally:
            overshoot_pct = ((rally['high_reached'] - rally['ended_at']) / rally['ended_at']) * 100
            if overshoot_pct > 5:  # >5% overshoot counts
                overshoots += 1
    
    overshoot_rate = (overshoots / total_rallies) * 100
    
    # Bereken verwachte overshoot range
    target_to_resistance = ((resistance_level - target) / target) * 100
    
    if overshoot_rate > 60:
        overshoot_expectation = f"{overshoot_rate:.0f}% chance price overshoots {resistance_level}"
        expected_range = [resistance_level, resistance_level * 1.05]
    else:
        overshoot_expectation = f"Only {overshoot_rate:.0f}% historical overshoot rate"
        expected_range = [target * 0.98, resistance_level]
    
    return {
        'target_price': target,
        'resistance_level': resistance_level,
        'historical_overshoot_rate': f"{overshoot_rate:.1f}%",
        'overshoots_detected': overshoots,
        'total_rallies_analyzed': total_rallies,
        'expectation': overshoot_expectation,
        'likely_range_if_rally': expected_range,
        'distance_to_resistance_pct': round(target_to_resistance, 2)
    }

## Example: DVLT 3-Month Analysis (Jan-Apr 2026)

Hieronder een real-world voorbeeld met DVLT data.

In [ ]:
# Initialize analyzer
analyzer = ThreeMonthAnalyzer()

# January 2026 — Rally naar $1.07
analyzer.add_price_data(
    month="January 2026",
    high=1.07,
    low=0.69,
    close=0.84,
    perf_pct=54.88,
    avg_volume=8_500_000,
    notable_events="Strong rally from $0.69 to $1.07"
)

analyzer.add_news_event(
    month="January 2026",
    date="2026-01-07",
    headline="Scilex announces positive Zomigone trial results",
    price_before=0.69,
    price_after=0.82
)

analyzer.add_insider_event(
    month="January 2026",
    date="2026-01-15",
    shares=2_500_000,
    price=0.88,
    entity="Scilex Board Member"
)

analyzer.add_insider_event(
    month="January 2026",
    date="2026-01-22",
    shares=5_000_000,
    price=0.72,
    entity="Scilex CFO"
)

# February 2026 — Crash na rally
analyzer.add_price_data(
    month="February 2026",
    high=0.87,
    low=0.64,
    close=0.69,
    perf_pct=-17.86,
    avg_volume=6_200_000,
    notable_events="Gave back most of Jan gains"
)

# March 2026 — Insider dump continued
analyzer.add_price_data(
    month="March 2026",
    high=0.85,
    low=0.60,
    close=0.75,
    perf_pct=8.70,
    avg_volume=7_100_000,
    notable_events="Volatile sideways, insider selling"
)

analyzer.add_insider_event(
    month="March 2026",
    date="2026-03-12",
    shares=10_000_000,
    price=0.63,
    entity="Scilex Holdings LLC"
)

# April 2026 (current month)
analyzer.add_price_data(
    month="April 2026",
    high=0.84,
    low=0.64,
    close=0.74,  # Current as of Apr 24
    perf_pct=-1.33,
    avg_volume=5_800_000,
    notable_events="Testing resistance at $0.83-0.85"
)

print("═══════════════════════════════════════════════════════════════")
print("3-MONTH COMPARISON ANALYSIS — DVLT")
print("═══════════════════════════════════════════════════════════════\n")

# Monthly Summary
print("MONTHLY SUMMARY:")
print(analyzer.get_monthly_summary().to_string(index=False))
print()

In [ ]:
# Key Levels Analysis
print("\nKEY LEVELS (3-Month Range):")
print("═══════════════════════════════════════════════════════════════")
levels = analyzer.identify_key_levels()

print(f"RESISTANCE LEVELS:")
for i, r in enumerate(levels['resistance'][:3], 1):
    print(f"  R{i}: ${r}")

print(f"\nSUPPORT LEVELS:")
for i, s in enumerate(levels['support'][:3], 1):
    print(f"  S{i}: ${s}")

print(f"\n3-MONTH AVERAGES:")
print(f"  Avg High:  ${levels['avg_high']}")
print(f"  Avg Low:   ${levels['avg_low']}")
print(f"  Midpoint:  ${levels['range_midpoint']}")
print()

In [ ]:
# Insider Correlation Analysis
print("\nINSIDER SELLING CORRELATION:")
print("═══════════════════════════════════════════════════════════════")
insider_analysis = analyzer.analyze_insider_correlation()

print(f"Total Insider Sells: {insider_analysis['total_insider_sells']}")
print(f"Avg Sell Price: ${insider_analysis['avg_sell_price']}")
print(f"Correlation: {insider_analysis['correlation_strength']}")
print(f"\nPrice dropped after {insider_analysis['price_dropped_after']} out of {insider_analysis['total_insider_sells']} sells")
print()

In [ ]:
# Rally Pattern Analysis
print("\nRALLY PATTERN ANALYSIS:")
print("═══════════════════════════════════════════════════════════════")
rally_patterns = analyzer.generate_rally_pattern_analysis()

print(f"Total Rallies Detected: {rally_patterns['total_rallies']}")
print(f"Average Rally Size: {rally_patterns['avg_rally_size']}%")
print(f"Average Giveback: {rally_patterns['avg_giveback']}%")
print(f"Crash Follow Rate: {rally_patterns['crash_follow_rate']}")
print("\nRALLY DETAILS:")
print(rally_patterns['details'].to_string(index=False))
print()

In [ ]:
# Entry Timing Assessment
print("\nENTRY TIMING ASSESSMENT:")
print("═══════════════════════════════════════════════════════════════")
CURRENT_DVLT_PRICE = 0.74
entry_timing = analyzer.assess_entry_timing(CURRENT_DVLT_PRICE)

print(f"Current Price: ${entry_timing['current_price']}")
print(f"Percentile Position: {entry_timing['percentile_position']}th percentile")
print(f"Timing Quality: {entry_timing['timing_quality']}")
print(f"\nNEAREST LEVELS:")
print(f"  Support: ${entry_timing['nearest_support']} ({entry_timing['support_distance_pct']}% below)")
print(f"  Resistance: ${entry_timing['nearest_resistance']} ({entry_timing['resistance_distance_pct']}% above)")
print(f"\nRisk/Reward Ratio: {entry_timing['risk_reward_ratio']}:1")
print(f"\n3-MONTH RANGE:")
print(f"  Low:  ${entry_timing['3m_range']['low']}")
print(f"  Mid:  ${entry_timing['3m_range']['midpoint']}")
print(f"  High: ${entry_timing['3m_range']['high']}")
print()

In [ ]:
# Overshoot Probability Analysis
print("\nOVERSHOOT PROBABILITY ANALYSIS:")
print("═══════════════════════════════════════════════════════════════")

# Extract historical rally data
rally_data = rally_patterns['details'].to_dict('records')

overshoot = calculate_overshoot_probability(
    target=0.83,  # User's breakeven price
    resistance_level=0.85,
    historical_rallies=rally_data
)

print(f"Target (Breakeven): ${overshoot['target_price']}")
print(f"Resistance Level: ${overshoot['resistance_level']}")
print(f"Distance to Resistance: {overshoot['distance_to_resistance_pct']}%")
print(f"\nHISTORICAL OVERSHOOT RATE:")
print(f"  Rate: {overshoot['historical_overshoot_rate']}")
print(f"  Overshoots: {overshoot['overshoots_detected']}/{overshoot['total_rallies_analyzed']} rallies")
print(f"\nEXPECTATION:")
print(f"  {overshoot['expectation']}")
print(f"  Likely Range: ${overshoot['likely_range_if_rally'][0]:.2f} - ${overshoot['likely_range_if_rally'][1]:.2f}")
print("\n═══════════════════════════════════════════════════════════════")

## Interactive: Your Own 3-Month Analysis

Pas de waardes hieronder aan voor jouw eigen stock analysis:

In [ ]:
# ==== PAS DEZE WAARDES AAN ====
MY_TICKER = "EXAMPLE"
MY_CURRENT_PRICE = 10.50

# Maak je eigen analyzer
my_analyzer = ThreeMonthAnalyzer()

# Voeg maanddata toe (pas aan naar jouw situatie)
my_analyzer.add_price_data(
    month="Month 1",
    high=11.50,
    low=9.00,
    close=10.00,
    perf_pct=5.0,
    avg_volume=1_000_000
)

my_analyzer.add_price_data(
    month="Month 2",
    high=10.80,
    low=9.50,
    close=10.20,
    perf_pct=2.0,
    avg_volume=800_000
)

my_analyzer.add_price_data(
    month="Month 3",
    high=11.00,
    low=10.00,
    close=10.50,
    perf_pct=2.94,
    avg_volume=900_000
)

# Run analysis
print(f"3-MONTH ANALYSIS: {MY_TICKER}")
print("=" * 60)
print(my_analyzer.get_monthly_summary().to_string(index=False))
print()

timing = my_analyzer.assess_entry_timing(MY_CURRENT_PRICE)
print(f"\nEntry Timing @ ${MY_CURRENT_PRICE}: {timing['timing_quality']}")
print(f"Percentile: {timing['percentile_position']}th")
print(f"R/R Ratio: {timing['risk_reward_ratio']}:1")